# Séance 4 · Exercices — Collecter et nettoyer les données · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

**Niveau de la séance : ⭐⭐ Intermédiaire** · chaque exercice porte son propre niveau (⭐ Débutant · ⭐⭐ Intermédiaire · ⭐⭐⭐ Avancé).

- Comment travailler : lis l'énoncé, code dans la cellule « À toi », lance la cellule de vérification (✅ / ❌), et n'ouvre la solution qu'après avoir vraiment essayé.
- Ce notebook tourne dans **Google Colab** : rien à installer.
- Clique sur une cellule et fais `Maj + Entrée` pour l'exécuter. Fais les exercices dans l'ordre : certains réutilisent les variables des précédents.


## Préparation

Mêmes données que la leçon : les 800 **Pokémon** (propres) et les 244 additions **Tips**. La fonction `abimer` de la leçon est recopiée ici : elle fabrique le tableau sale `df` sur lequel tu vas travailler, avec les mêmes dégâts pour tout le monde (`seed=42`).

Le **journal des corrections** (`journal` + `noter`) est ton livrable : une ligne par réparation.

In [ ]:
import pandas as pd
import numpy as np
import requests
import difflib

URL_POKEMON = "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"
URL_TIPS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
try:
    pokemon_propre = pd.read_csv(URL_POKEMON)
    tips_propre = pd.read_csv(URL_TIPS)
    print("Pokémon propre :", pokemon_propre.shape, "· Tips propre :", tips_propre.shape)
except Exception as erreur:
    print("Pas de réseau ? Impossible de charger les fichiers :", erreur)

MOIS_FR = ["janv", "fév", "mars", "avr", "mai", "juin", "juil", "août", "sept", "oct", "nov", "déc"]

def abimer(df, col_categorie, col_nombre, seed=42):
    # La même fonction que dans la leçon : trous, fautes de frappe, dates mélangées, nombres en texte, doublons
    rng = np.random.default_rng(seed)
    sale = df.copy()
    for col in [col_categorie, col_nombre]:
        lignes = rng.choice(sale.index, size=len(sale) // 20, replace=False)
        sale.loc[lignes, col] = np.nan
    def faute(v):
        if not isinstance(v, str): return v
        r = rng.random()
        if r < 0.04: return v.lower()
        if r < 0.08: return v + " "
        if r < 0.11: return v[:-2] + v[-1] + v[-2]
        return v
    sale[col_categorie] = sale[col_categorie].apply(faute)
    jours = pd.Timestamp("2024-01-01") + pd.to_timedelta(rng.integers(0, 365, len(sale)), unit="D")
    def ecrire_date(d, style):
        if style == 0: return d.strftime("%Y-%m-%d")
        if style == 1: return d.strftime("%d/%m/%Y")
        return f"{d.day} {MOIS_FR[d.month - 1]} {d.year}"
    sale["date_capture"] = [ecrire_date(d, s) for d, s in zip(jours, rng.integers(0, 3, len(sale)))]
    sale[col_nombre] = sale[col_nombre].map(lambda v: v if pd.isna(v) else f"{v:g} ")
    sale = pd.concat([sale, sale.sample(10, random_state=seed)])
    return sale.sample(frac=1, random_state=seed).reset_index(drop=True)

df = abimer(pokemon_propre, col_categorie="Type 1", col_nombre="HP")   # le tableau sale à réparer
print("Tableau abîmé df :", df.shape)

journal = []   # le journal des corrections : une ligne par réparation

def noter(message):
    journal.append(message)
    print("📝", message)

def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais lever d'exception (condition = booléen, ou fonction sans argument)."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as erreur:
        print(f"❌ {nom} : erreur pendant la vérification → {erreur}")
        return False
    print(f"✅ {nom}" if ok else f"❌ {nom} : pas encore, réessaie !")
    return ok


## Exercice 1 ⭐ · Le bilan de santé

Avant de réparer quoi que ce soit, on fait le diagnostic. Remplis `nb_lignes`, `nb_colonnes` et `cases_vides`
(le nombre total de cases vides dans **tout** le tableau `df`).

Résultat attendu : plus de 800 lignes (le fichier propre en a 800) et plusieurs centaines de cases vides.

<details><summary>Indice</summary>

`df.shape` donne (lignes, colonnes) ; `df.isna().sum()` compte les vides par colonne, et un second `.sum()` fait le total.

</details>

In [ ]:
# À toi
nb_lignes = None
nb_colonnes = None
cases_vides = None
print(nb_lignes, "lignes ·", nb_colonnes, "colonnes ·", cases_vides, "cases vides")

In [ ]:
verifier("Exercice 1 · lignes", nb_lignes == 810)
verifier("Exercice 1 · colonnes", nb_colonnes == 14)
verifier("Exercice 1 · cases vides", cases_vides == 471)

<details><summary>Solution</summary>

```python
nb_lignes, nb_colonnes = df.shape
cases_vides = int(df.isna().sum().sum())
print(nb_lignes, "lignes ·", nb_colonnes, "colonnes ·", cases_vides, "cases vides")
```

</details>

## Exercice 2 ⭐ · Les doublons

Compte les lignes en double dans `nb_doublons`, supprime-les de `df` (sans oublier `reset_index(drop=True)`),
puis écris la correction dans le journal avec `noter(...)`.

Résultat attendu : il reste exactement 800 lignes.

<details><summary>Indice</summary>

`df.duplicated().sum()` compte, `df.drop_duplicates()` enlève. Le journal : `noter(f"Doublons : {nb_doublons} lignes supprimées")`.

</details>

In [ ]:
# À toi
nb_doublons = None
# df = ...
# noter(...)
print("Doublons :", nb_doublons, "· lignes restantes :", len(df))

In [ ]:
verifier("Exercice 2 · doublons comptés", nb_doublons == 10)
verifier("Exercice 2 · doublons supprimés", len(df) == 800 and df.duplicated().sum() == 0)
verifier("Exercice 2 · journal", len(journal) >= 1)

<details><summary>Solution</summary>

```python
nb_doublons = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
noter(f"Doublons : {nb_doublons} lignes identiques supprimées, il reste {len(df)} lignes")
print("Doublons :", nb_doublons, "· lignes restantes :", len(df))
```

</details>

## Exercice 3 ⭐ · Où sont les trous ?

Construis `vides`, le nombre de cases vides **par colonne**, et mets dans `colonne_la_plus_vide` le nom de la colonne
qui en a le plus. Regarde ensuite `Type 2` : est-ce une erreur à réparer, ou un vide qui a un sens ?

Résultat attendu : `vides["HP"]` vaut 40.

<details><summary>Indice</summary>

`df.isna().sum()` renvoie une Series ; `.idxmax()` donne le nom (l'index) de la plus grande valeur.

</details>

In [ ]:
# À toi
vides = None
colonne_la_plus_vide = None
print(vides)
print("Colonne la plus vide :", colonne_la_plus_vide)

In [ ]:
verifier("Exercice 3 · vides par colonne", vides is not None and int(vides["HP"]) == 40 and int(vides["Type 1"]) == 40)
verifier("Exercice 3 · colonne la plus vide", colonne_la_plus_vide == "Type 2")

<details><summary>Solution</summary>

```python
vides = df.isna().sum()
colonne_la_plus_vide = vides.idxmax()
print(vides)
print("Colonne la plus vide :", colonne_la_plus_vide)
# Type 2 vide = « pas de second type » : ce n'est pas une erreur, on le garde tel quel.
```

</details>

## Exercice 4 ⭐ · Espaces et majuscules

`fire`, `Fire ` et `Fire` sont trois valeurs différentes pour l'ordinateur. Mesure `avant` (le nombre de valeurs
différentes de `Type 1`), normalise la colonne (espaces enlevés, majuscule au début, le reste en minuscules),
mesure `apres`, et note la correction dans le journal.

Résultat attendu : on passe de 56 valeurs différentes à 30 (il restera les lettres inversées, pour l'exercice suivant).

<details><summary>Indice</summary>

`.nunique()` compte les valeurs différentes ; `.str.strip()` puis `.str.capitalize()`.

</details>

In [ ]:
# À toi
avant = None
# df["Type 1"] = ...
apres = None
print("Avant :", avant, "· après :", apres)
print(sorted(df["Type 1"].dropna().unique()))

In [ ]:
verifier("Exercice 4 · avant", avant == 56)
verifier("Exercice 4 · après", apres == 30 and df["Type 1"].nunique() == 30)
verifier("Exercice 4 · journal", len(journal) >= 2)

<details><summary>Solution</summary>

```python
avant = df["Type 1"].nunique()
df["Type 1"] = df["Type 1"].str.strip().str.capitalize()
apres = df["Type 1"].nunique()
noter(f"Type 1 : espaces et majuscules normalisés ({avant} → {apres} valeurs différentes)")
print("Avant :", avant, "· après :", apres)
print(sorted(df["Type 1"].dropna().unique()))
```

</details>

## Exercice 5 ⭐⭐ · Le mapping des fautes de frappe

Il reste des lettres inversées (`Fier`, `Watre`...). Construis la liste `suspects` (les valeurs de `Type 1` qui ne
sont pas dans `types_valides`), puis le dictionnaire `corrections` (valeur fausse → valeur juste), applique-le à la
colonne avec `replace`, et note la correction.

Résultat attendu : 12 suspects, 12 corrections, et plus aucune valeur hors de `types_valides` (à part les vides).

<details><summary>Indice</summary>

Tu peux écrire le dictionnaire à la main, ou laisser `difflib.get_close_matches(faux, types_valides, n=1, cutoff=0.6)` trouver le mot valide le plus proche.

</details>

In [ ]:
# À toi
types_valides = sorted(pokemon_propre["Type 1"].unique())
suspects = None
corrections = {}
# df["Type 1"] = ...
print("Suspects :", suspects)
print("Corrections :", corrections)

In [ ]:
verifier("Exercice 5 · suspects", suspects is not None and len(suspects) == 12)
verifier("Exercice 5 · corrections", len(corrections) == 12 and all(v in types_valides for v in corrections.values()))
verifier("Exercice 5 · plus de suspect", set(df["Type 1"].dropna()) <= set(types_valides))

<details><summary>Solution</summary>

```python
types_valides = sorted(pokemon_propre["Type 1"].unique())
suspects = [t for t in df["Type 1"].dropna().unique() if t not in types_valides]
corrections = {}
for faux in suspects:
    proche = difflib.get_close_matches(faux, types_valides, n=1, cutoff=0.6)
    corrections[faux] = proche[0] if proche else faux
df["Type 1"] = df["Type 1"].replace(corrections)
noter(f"Type 1 : {len(corrections)} fautes de frappe corrigées avec un mapping → {df['Type 1'].nunique()} types valides")
print("Suspects :", suspects)
print("Corrections :", corrections)
```

</details>

## Exercice 6 ⭐⭐ · Des nombres, pas du texte

La colonne `HP` contient des textes comme `"45 "`. Garde son type actuel dans `type_avant`, convertis-la en vrais
nombres (à virgule, car il y a encore des vides), calcule `hp_moyen` (arrondi à 1 décimale) et note la correction.

Résultat attendu : `type_avant` est `object`, `hp_moyen` vaut 69.3.

<details><summary>Indice</summary>

`df["HP"].dtype` ; puis `.str.strip().astype(float)`. Pourquoi pas `int` ? Un `NaN` ne peut pas vivre dans une colonne d'entiers.

</details>

In [ ]:
# À toi
type_avant = None
# df["HP"] = ...
hp_moyen = None
print("Type avant :", type_avant, "· type après :", df["HP"].dtype, "· HP moyen :", hp_moyen)

In [ ]:
verifier("Exercice 6 · type avant", str(type_avant) == "object")
verifier("Exercice 6 · conversion", pd.api.types.is_float_dtype(df["HP"]))
verifier("Exercice 6 · moyenne", hp_moyen == 69.3)

<details><summary>Solution</summary>

```python
type_avant = df["HP"].dtype
df["HP"] = df["HP"].str.strip().astype(float)
hp_moyen = round(df["HP"].mean(), 1)
noter("HP : texte converti en nombre (strip + astype(float))")
print("Type avant :", type_avant, "· type après :", df["HP"].dtype, "· HP moyen :", hp_moyen)
```

</details>

## Exercice 7 ⭐⭐ · Remplir ou supprimer ?

Deux colonnes ont des trous. Pour `HP` : calcule `mediane_hp`, remplis les vides avec, puis repasse la colonne en
entiers. Pour `Type 1` : remplis les vides par `"Inconnu"` (inventer un type serait pire) et compte-les dans `nb_inconnus`.
Une ligne de journal par colonne.

Résultat attendu : médiane 65, 40 « Inconnu », plus aucun vide dans ces deux colonnes.

<details><summary>Indice</summary>

`.median()`, `.fillna(valeur)`, `.astype(int)` ; `(df["Type 1"] == "Inconnu").sum()`.

</details>

In [ ]:
# À toi
mediane_hp = None
# df["HP"] = ...
# df["Type 1"] = ...
nb_inconnus = None
print("Médiane HP :", mediane_hp, "· Inconnus :", nb_inconnus)
print(df[["Type 1", "HP"]].isna().sum())

In [ ]:
verifier("Exercice 7 · médiane", mediane_hp == 65)
verifier("Exercice 7 · HP rempli en entiers", df["HP"].isna().sum() == 0 and pd.api.types.is_integer_dtype(df["HP"]))
verifier("Exercice 7 · Type 1 rempli", nb_inconnus == 40 and df["Type 1"].isna().sum() == 0)
verifier("Exercice 7 · journal", len(journal) >= 5)

<details><summary>Solution</summary>

```python
mediane_hp = df["HP"].median()
nb_vides_hp = int(df["HP"].isna().sum())
df["HP"] = df["HP"].fillna(mediane_hp).astype(int)
noter(f"HP : {nb_vides_hp} cases vides remplies par la médiane ({mediane_hp:g}), colonne repassée en entiers")
nb_inconnus = int(df["Type 1"].isna().sum())
df["Type 1"] = df["Type 1"].fillna("Inconnu")
noter(f"Type 1 : {nb_inconnus} cases vides remplacées par 'Inconnu'")
print("Médiane HP :", mediane_hp, "· Inconnus :", nb_inconnus)
print(df[["Type 1", "HP"]].isna().sum())
```

</details>

## Exercice 8 ⭐⭐ · Trois formats de date

`date_capture` mélange `2024-01-05`, `05/01/2024` et `5 janv 2024`. Convertis la colonne en vraies dates
(essaie chaque format avec `errors="coerce"`, puis recolle les résultats avec `fillna`). Compte dans `dates_non_lues`
les dates restées vides (`NaT`), et mets dans `mois_record` le numéro du mois qui compte le plus de captures.

Résultat attendu : 0 date non lue, mois record = 5 (mai).

<details><summary>Indice</summary>

Pour le format en lettres, remplace d'abord chaque mois de `MOIS_FR` par son numéro (`str.replace`), puis lis avec `format="%d %m %Y"`. Ensuite `.dt.month.value_counts().idxmax()`.

</details>

In [ ]:
# À toi
texte = df["date_capture"]
# essai_iso = pd.to_datetime(texte, format="%Y-%m-%d", errors="coerce")
# ...
# df["date_capture"] = ...
dates_non_lues = None
mois_record = None
print("Dates non lues :", dates_non_lues, "· mois record :", mois_record)

In [ ]:
verifier("Exercice 8 · vraies dates", pd.api.types.is_datetime64_any_dtype(df["date_capture"]))
verifier("Exercice 8 · tout est lu", dates_non_lues == 0 and df["date_capture"].isna().sum() == 0)
verifier("Exercice 8 · mois record", mois_record == 5)

<details><summary>Solution</summary>

```python
texte = df["date_capture"]
essai_iso = pd.to_datetime(texte, format="%Y-%m-%d", errors="coerce")
essai_fr = pd.to_datetime(texte, format="%d/%m/%Y", errors="coerce")
texte_mois = texte.copy()
for numero, mois in enumerate(MOIS_FR, start=1):
    texte_mois = texte_mois.str.replace(mois, str(numero), regex=False)
essai_lettres = pd.to_datetime(texte_mois, format="%d %m %Y", errors="coerce")
df["date_capture"] = essai_iso.fillna(essai_fr).fillna(essai_lettres)
dates_non_lues = int(df["date_capture"].isna().sum())
mois_record = int(df["date_capture"].dt.month.value_counts().idxmax())
noter("date_capture : 3 formats convertis en vraies dates (to_datetime)")
print("Dates non lues :", dates_non_lues, "· mois record :", mois_record)
```

</details>

## Exercice 9 ⭐⭐ · Prouver son travail

Un bon nettoyage se prouve. Compare `df` au tableau propre d'origine : `memes_noms` (`True` si les deux tableaux
contiennent exactement les mêmes Pokémon, colonne `Name`), `ecart_hp` (l'écart absolu entre les deux moyennes de `HP`,
arrondi à 2 décimales). Puis affiche le journal, numéroté.

Résultat attendu : `memes_noms` vaut `True`, `ecart_hp` est inférieur à 0.5, le journal a au moins 6 lignes.

<details><summary>Indice</summary>

`set(df["Name"]) == set(pokemon_propre["Name"])` ; `abs(a - b)` ; `for i, ligne in enumerate(journal, start=1)`.

</details>

In [ ]:
# À toi
memes_noms = None
ecart_hp = None
print("Mêmes Pokémon :", memes_noms, "· écart de HP moyen :", ecart_hp)
print("=== Journal des corrections ===")

In [ ]:
verifier("Exercice 9 · mêmes Pokémon", memes_noms is True or memes_noms == True)
verifier("Exercice 9 · écart de HP", ecart_hp is not None and ecart_hp < 0.5)
verifier("Exercice 9 · journal complet", len(journal) >= 6)

<details><summary>Solution</summary>

```python
memes_noms = set(df["Name"]) == set(pokemon_propre["Name"])
ecart_hp = round(abs(df["HP"].mean() - pokemon_propre["HP"].mean()), 2)
print("Mêmes Pokémon :", memes_noms, "· écart de HP moyen :", ecart_hp)
print("=== Journal des corrections ===")
for i, ligne in enumerate(journal, start=1):
    print(f"{i}. {ligne}")
```

</details>

## Exercice 10 ⭐⭐⭐ · Une API avec filet de sécurité

Écris la fonction `demander(url, secours)` : elle appelle l'API avec `requests.get` (`timeout=10`), vérifie la
réponse avec `raise_for_status()` et renvoie le JSON ; si **quoi que ce soit** échoue (pas de réseau, Pokémon
inexistant...), elle affiche un message et renvoie `secours`. Puis demande la fiche d'Évoli (`eevee`) à la PokéAPI
et extrais `types_eevee` (la liste de ses types) et `hp_eevee` (sa stat `hp`).

Résultat attendu : `["normal"]` et `55` (avec ou sans réseau, grâce au dict de secours). La 3e vérification demande le réseau.

<details><summary>Indice</summary>

`try: ... except Exception as erreur: ...`. Les types : `[t["type"]["name"] for t in eevee["types"]]` ; les stats : un dictionnaire `{s["stat"]["name"]: s["base_stat"] for s in eevee["stats"]}`.

</details>

In [ ]:
# À toi
SECOURS_EEVEE = {"name": "eevee", "height": 3, "weight": 65,
                 "types": [{"type": {"name": "normal"}}],
                 "stats": [{"stat": {"name": "hp"}, "base_stat": 55}, {"stat": {"name": "attack"}, "base_stat": 55},
                           {"stat": {"name": "defense"}, "base_stat": 50}, {"stat": {"name": "speed"}, "base_stat": 55}]}

def demander(url, secours):
    # 1. requests.get(url, timeout=10)   2. .raise_for_status()   3. return .json()   4. except → renvoyer secours
    return secours   # ← remplace par ta version

eevee = demander("https://pokeapi.co/api/v2/pokemon/eevee", SECOURS_EEVEE)
types_eevee = None
hp_eevee = None
print(eevee["name"], "· types :", types_eevee, "· HP :", hp_eevee)

In [ ]:
verifier("Exercice 10 · types", types_eevee == ["normal"])
verifier("Exercice 10 · HP", hp_eevee == 55)
verifier("Exercice 10 · l'API répond vraiment (réseau)", lambda: demander("https://pokeapi.co/api/v2/pokemon/ditto", {"name": "secours"})["name"] == "ditto")
verifier("Exercice 10 · le filet de sécurité", lambda: demander("https://pokeapi.co/api/v2/pokemon/nexistepas-0000", {"name": "secours"})["name"] == "secours")

<details><summary>Solution</summary>

```python
SECOURS_EEVEE = {"name": "eevee", "height": 3, "weight": 65,
                 "types": [{"type": {"name": "normal"}}],
                 "stats": [{"stat": {"name": "hp"}, "base_stat": 55}, {"stat": {"name": "attack"}, "base_stat": 55},
                           {"stat": {"name": "defense"}, "base_stat": 50}, {"stat": {"name": "speed"}, "base_stat": 55}]}

def demander(url, secours):
    try:
        reponse = requests.get(url, timeout=10)
        reponse.raise_for_status()          # erreur si le site répond autre chose que 200 OK
        return reponse.json()
    except Exception as erreur:
        print("Pas de réseau ou URL invalide ? On utilise les données de secours.", erreur)
        return secours

eevee = demander("https://pokeapi.co/api/v2/pokemon/eevee", SECOURS_EEVEE)
types_eevee = [t["type"]["name"] for t in eevee["types"]]
stats_eevee = {s["stat"]["name"]: s["base_stat"] for s in eevee["stats"]}
hp_eevee = stats_eevee["hp"]
print(eevee["name"], "· types :", types_eevee, "· HP :", hp_eevee)
```

</details>

## Exercice 11 ⭐⭐⭐ · La météo de trois villes

Avec Open-Meteo (sans clé) et ta fonction `demander`, construis le DataFrame `meteo_villes` : une ligne par ville,
les colonnes `ville`, `temperature` et `vent` (dans cet ordre). Puis mets dans `ville_la_plus_chaude` le nom de la ville
la plus chaude en ce moment.

Résultat attendu : un tableau de 3 lignes × 3 colonnes, trié ou non, et un nom de ville.

<details><summary>Indice</summary>

Une boucle `for ville, (lat, lon) in villes.items():` qui construit l'URL avec un f-string, appelle `demander(url, SECOURS_METEO)["current_weather"]` et ajoute un dict à une liste ; `pd.DataFrame(liste)` à la fin. Le plus chaud : `.sort_values("temperature", ascending=False).iloc[0]["ville"]`.

</details>

In [ ]:
# À toi
villes = {"Paris": (48.85, 2.35), "Marseille": (43.30, 5.37), "Lille": (50.63, 3.06)}
SECOURS_METEO = {"current_weather": {"temperature": 18.0, "windspeed": 12.0, "winddirection": 200, "weathercode": 3, "time": "2024-06-01T12:00"}}

meteo_villes = None
ville_la_plus_chaude = None
print(meteo_villes)
print("La plus chaude :", ville_la_plus_chaude)

In [ ]:
verifier("Exercice 11 · un DataFrame 3 × 3", isinstance(meteo_villes, pd.DataFrame) and list(meteo_villes.columns) == ["ville", "temperature", "vent"] and len(meteo_villes) == 3)
verifier("Exercice 11 · les trois villes", isinstance(meteo_villes, pd.DataFrame) and set(meteo_villes["ville"]) == set(villes))
verifier("Exercice 11 · des nombres", isinstance(meteo_villes, pd.DataFrame) and pd.api.types.is_numeric_dtype(meteo_villes["temperature"]))
verifier("Exercice 11 · la plus chaude", ville_la_plus_chaude in villes)

<details><summary>Solution</summary>

```python
villes = {"Paris": (48.85, 2.35), "Marseille": (43.30, 5.37), "Lille": (50.63, 3.06)}
SECOURS_METEO = {"current_weather": {"temperature": 18.0, "windspeed": 12.0, "winddirection": 200, "weathercode": 3, "time": "2024-06-01T12:00"}}

lignes = []
for ville, (lat, lon) in villes.items():
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    actuel = demander(url, SECOURS_METEO)["current_weather"]
    lignes.append({"ville": ville, "temperature": actuel["temperature"], "vent": actuel["windspeed"]})
meteo_villes = pd.DataFrame(lignes).sort_values("temperature", ascending=False).reset_index(drop=True)
ville_la_plus_chaude = meteo_villes.iloc[0]["ville"]
print(meteo_villes)
print("La plus chaude :", ville_la_plus_chaude)
```

</details>

## Exercice 12 ⭐⭐⭐ · Défi : nettoie Tips de A à Z

Le tableau `tips` (244 additions d'un restaurant) a été abîmé avec la même fonction. Applique toute la recette,
dans l'ordre, en tenant ton journal `journal_tips` avec `noter_tips` :

1. doublons → 2. catégorie `day` (espaces, majuscules, mapping vers `jours_valides`) → 3. `total_bill` en nombre →
4. valeurs manquantes (`day` et `total_bill` : supprimer ou remplir ? **justifie dans le journal**) → 5. `date_capture` en vraies dates.

Résultat attendu : plus aucun doublon, entre 232 et 244 lignes, des jours valides (+ éventuellement `"Inconnu"`),
`total_bill` numérique sans vide, des dates lues, et au moins 5 lignes de journal.

<details><summary>Indice</summary>

Recopie la logique des exercices 2 à 8 en changeant les noms de colonnes. Pour le mapping des jours, `difflib.get_close_matches(faux, jours_valides, n=1, cutoff=0.5)` (les mots sont courts, on baisse le seuil).

</details>

In [ ]:
# À toi
tips = abimer(tips_propre, col_categorie="day", col_nombre="total_bill")
jours_valides = sorted(tips_propre["day"].unique())
journal_tips = []

def noter_tips(message):
    journal_tips.append(message)
    print("📝", message)

noter_tips(f"Départ : {len(tips)} lignes, {tips.isna().sum().sum()} cases vides")

# 1. doublons

# 2. catégorie day

# 3. total_bill en nombre

# 4. valeurs manquantes (justifie ton choix)

# 5. dates

for i, ligne in enumerate(journal_tips, start=1):
    print(f"{i}. {ligne}")

In [ ]:
verifier("Défi · doublons et lignes", tips.duplicated().sum() == 0 and 232 <= len(tips) <= 244)
verifier("Défi · jours valides", set(tips["day"].dropna()) <= set(jours_valides) | {"Inconnu"} and tips["day"].isna().sum() == 0)
verifier("Défi · total_bill numérique", pd.api.types.is_numeric_dtype(tips["total_bill"]) and tips["total_bill"].isna().sum() == 0)
verifier("Défi · moyenne cohérente", lambda: pd.api.types.is_numeric_dtype(tips["total_bill"]) and abs(tips["total_bill"].mean() - tips_propre["total_bill"].mean()) < 1.5)
verifier("Défi · dates", pd.api.types.is_datetime64_any_dtype(tips["date_capture"]) and tips["date_capture"].isna().sum() == 0)
verifier("Défi · journal", len(journal_tips) >= 5)

<details><summary>Solution</summary>

```python
tips = abimer(tips_propre, col_categorie="day", col_nombre="total_bill")
jours_valides = sorted(tips_propre["day"].unique())
journal_tips = []

def noter_tips(message):
    journal_tips.append(message)
    print("📝", message)

noter_tips(f"Départ : {len(tips)} lignes, {tips.isna().sum().sum()} cases vides")

# 1. doublons
n = int(tips.duplicated().sum())
tips = tips.drop_duplicates().reset_index(drop=True)
noter_tips(f"Doublons : {n} lignes supprimées, il reste {len(tips)}")

# 2. catégorie day
tips["day"] = tips["day"].str.strip().str.capitalize()
suspects = [d for d in tips["day"].dropna().unique() if d not in jours_valides]
corrections = {f: difflib.get_close_matches(f, jours_valides, n=1, cutoff=0.5)[0] for f in suspects}
tips["day"] = tips["day"].replace(corrections)
noter_tips(f"day : espaces et majuscules normalisés, {len(corrections)} fautes corrigées {corrections}")

# 3. total_bill en nombre
tips["total_bill"] = tips["total_bill"].str.strip().astype(float)
noter_tips("total_bill : texte converti en nombre")

# 4. valeurs manquantes : une addition sans montant ne sert à rien → on supprime ; un jour inconnu → 'Inconnu'
n = int(tips["total_bill"].isna().sum())
tips = tips.dropna(subset=["total_bill"]).reset_index(drop=True)
tips["day"] = tips["day"].fillna("Inconnu")
noter_tips(f"total_bill : {n} lignes sans montant supprimées (inutilisables) ; day vide → 'Inconnu' (on ne devine pas)")

# 5. dates
t = tips["date_capture"]
iso = pd.to_datetime(t, format="%Y-%m-%d", errors="coerce")
fr = pd.to_datetime(t, format="%d/%m/%Y", errors="coerce")
tm = t.copy()
for numero, mois in enumerate(MOIS_FR, start=1):
    tm = tm.str.replace(mois, str(numero), regex=False)
lettres = pd.to_datetime(tm, format="%d %m %Y", errors="coerce")
tips["date_capture"] = iso.fillna(fr).fillna(lettres)
noter_tips("date_capture : 3 formats convertis en vraies dates")

for i, ligne in enumerate(journal_tips, start=1):
    print(f"{i}. {ligne}")
```

</details>

## Bravo !

Tu as fait le tour du nettoyage : doublons, fautes de frappe, nombres en texte, valeurs manquantes, dates, et tu sais
aller chercher des données avec une API sans que ton programme plante. Pour aller plus loin : sauvegarde ton tableau
propre (`df.to_csv("pokemon_propre.csv", index=False)`) et relis ton journal comme si tu le découvrais dans 6 mois.
